# Tratamento do COTAHIST da B3 — Geração de Séries Brutas por Ticker

**Apêndice — TCC (otimização de carteiras, B3).**

Este notebook lê os arquivos anuais `COTAHIST.AAAA.TXT` da B3 (2010–2025), faz o *parsing* do **Registro 01** (largura fixa, 245 bytes) conforme a layout oficial, filtra os **ativos de ação negociados no mercado à vista** e grava, para **cada ticker**, um `.csv` e um `.parquet` dentro de uma pasta com o nome do ticker.

**Por que esta base e não a da Economática:** o COTAHIST traz o **preço como negociado no dia** (`PREULT`), *sem* ajuste retroativo de proventos/grupamentos — é o preço nominal bruto necessário para a tela de investabilidade (piso de R$ 1 só faz sentido sobre o preço unitário real). Além disso, o campo **`CODBDI`** carimba oficialmente, dia a dia, quando um papel está em situação especial (recuperação judicial, concordata, RAET, intervenção) — um sinal *point-in-time*, sem viés de sobrevivência, para a máscara.

> A série **ajustada** da Economática continua sendo a fonte de **retornos/otimização**. Esta base bruta serve à **máscara de investabilidade**, ligada por `CODNEG` (ticker) + data, com `CODISI` (ISIN) disponível para reconciliar trocas de código (ex.: `GOLL4`→`GOLL54`).

In [1]:
from pathlib import Path

# ============================ CONFIGURAÇÃO ============================
# Ajuste apenas estas constantes ao seu ambiente.
PASTA_ENTRADA = Path(r"C:\VSCodeWorkspace\1_TCC_Final\data\B3_COTAHIST_2010_2025")

# ATENÇÃO: você escreveu a pasta como "Tratatos". Se for "Tratados", corrija aqui:
PASTA_SAIDA   = Path(r"C:\VSCodeWorkspace\1_TCC_Final\data\B3_COTAHIST_Tratatos")

ANO_INI, ANO_FIM = 2010, 2025      # filtra registros pelo ano do pregão
TPMERC_MANTIDOS  = {"010"}          # 010 = mercado à vista (apenas)
GERAR_PARQUET    = True             # o .csv é sempre gerado
ENCODING         = "latin-1"        # encoding padrão dos arquivos COTAHIST
# =====================================================================

## 1. Layout do Registro 01 e regras de classificação

Posições (base 0, semiabertas) de cada campo do Registro 01, conforme a layout COTAHIST. Preços no formato `(11)V99` e o volume financeiro `(16)V99` vêm **sem o ponto decimal** → dividir por 100. `PTOEXE` é `(07)V06` → dividir por 1.000.000. `TOTNEG`, `QUATOT`, `FATCOT` são inteiros.

In [2]:
import pandas as pd
import numpy as np
import re

# Registro 01 — COTAHIST (245 bytes). (início, fim) base-0, semiaberto.
SLICES = {
    'TIPREG': (0, 2),    'DATA':   (2, 10),   'CODBDI': (10, 12),  'CODNEG': (12, 24),
    'TPMERC': (24, 27),  'NOMRES': (27, 39),  'ESPECI': (39, 49),  'PRAZOT': (49, 52),
    'MODREF': (52, 56),  'PREABE': (56, 69),  'PREMAX': (69, 82),  'PREMIN': (82, 95),
    'PREMED': (95, 108), 'PREULT': (108, 121),'PREOFC': (121, 134),'PREOFV': (134, 147),
    'TOTNEG': (147, 152),'QUATOT': (152, 170),'VOLTOT': (170, 188),'PREEXE': (188, 201),
    'INDOPC': (201, 202),'DATVEN': (202, 210),'FATCOT': (210, 217),'PTOEXE': (217, 230),
    'CODISI': (230, 242),'DISMES': (242, 245),
}

# Campos monetários em (11)V99 e volume financeiro (16)V99 -> dividir por 100
CAMPOS_PRECO_V99 = ['PREABE','PREMAX','PREMIN','PREMED','PREULT','PREOFC','PREOFV','PREEXE','VOLTOT']
CAMPOS_TEXTO     = ['CODNEG','CODBDI','TPMERC','NOMRES','ESPECI','PRAZOT','MODREF','CODISI','INDOPC','DISMES']

# CODBDI de situação especial (a B3 marca o papel no dia): point-in-time, sem viés de sobrevivência
CODBDI_SITUACAO = {'06','07','08','09','11'}  # 06=concordata 07=RJ extra 08=RJ 09=RAET 11=intervenção

def is_acao(especi: str) -> bool:
    """Mantém apenas ON/PN/UNT (e variantes), descartando direitos, recibos, bônus, BDR, fundos etc."""
    e = (especi or '').strip().upper()
    if not e:
        return False
    if 'REC' in e or 'DIR' in e or 'BNS' in e:   # recibos/direitos/bônus de subscrição
        return False
    return e.startswith(('ON', 'OR', 'PN', 'PR', 'UNT'))

# Sanidade do layout: os 26 campos devem cobrir exatamente 245 bytes, sem buracos
_p = 0
for _k, (_a, _b) in SLICES.items():
    assert _a == _p, f"gap/sobreposição antes de {_k}"
    _p = _b
assert _p == 245, f"layout não soma 245 bytes: {_p}"
print("Layout OK: 26 campos, 245 bytes.")

Layout OK: 26 campos, 245 bytes.


## 2. Leitura e *parsing* dos arquivos anuais

Varredura linha a linha (streaming, baixo uso de memória): mantém só `TIPREG=01`, `TPMERC` à vista e `ESPECI` de ação. Lê **qualquer `.txt`** na pasta de entrada, independentemente do nome exato do arquivo.

In [3]:
if not PASTA_ENTRADA.exists():
    raise FileNotFoundError(f"Pasta de entrada não encontrada: {PASTA_ENTRADA}")

arquivos = sorted(p for p in PASTA_ENTRADA.glob('*') if p.suffix.lower() == '.txt')
if not arquivos:
    raise FileNotFoundError(f"Nenhum arquivo .txt encontrado em {PASTA_ENTRADA}")
print(f"{len(arquivos)} arquivo(s) .txt encontrados.\n")

COLS = list(SLICES.keys())
_pos = list(SLICES.values())
registros = []

for arq in arquivos:
    n0 = len(registros)
    with open(arq, encoding=ENCODING, errors='replace') as fh:
        for raw in fh:
            ln = raw.rstrip('\r\n')
            if len(ln) < 245 or ln[0:2] != '01':
                continue
            if ln[24:27] not in TPMERC_MANTIDOS:          # TPMERC
                continue
            if not is_acao(ln[39:49]):                    # ESPECI
                continue
            registros.append(tuple(ln[a:b] for (a, b) in _pos))
    print(f"  {arq.name}: +{len(registros) - n0:,} registros de acao (vista)")

bruto = pd.DataFrame(registros, columns=COLS)
print(f"\nTotal de registros brutos mantidos: {len(bruto):,}")

16 arquivo(s) .txt encontrados.

  COTAHIST_A2010.TXT: +81,861 registros de acao (vista)
  COTAHIST_A2011.TXT: +81,914 registros de acao (vista)
  COTAHIST_A2012.TXT: +77,683 registros de acao (vista)
  COTAHIST_A2013.TXT: +77,222 registros de acao (vista)
  COTAHIST_A2014.TXT: +76,278 registros de acao (vista)
  COTAHIST_A2015.TXT: +71,523 registros de acao (vista)
  COTAHIST_A2016.TXT: +71,259 registros de acao (vista)
  COTAHIST_A2017.TXT: +75,071 registros de acao (vista)
  COTAHIST_A2018.TXT: +76,727 registros de acao (vista)
  COTAHIST_A2019.TXT: +81,040 registros de acao (vista)
  COTAHIST_A2020.TXT: +88,543 registros de acao (vista)
  COTAHIST_A2021.TXT: +99,571 registros de acao (vista)
  COTAHIST_A2022.TXT: +98,367 registros de acao (vista)
  COTAHIST_A2023.TXT: +92,610 registros de acao (vista)
  COTAHIST_A2024.TXT: +91,421 registros de acao (vista)
  COTAHIST_A2025.TXT: +88,339 registros de acao (vista)

Total de registros brutos mantidos: 1,329,429


## 3. Tipagem, escala decimal e colunas derivadas

- `DATA_PREGAO` (datetime) a partir de `AAAAMMDD`.
- Preços e `VOLTOT` ÷ 100; `PTOEXE` ÷ 1.000.000; `TOTNEG`/`QUATOT`/`FATCOT` inteiros.
- **`PRECO_UNIT` = `PREULT` ÷ `FATCOT`** → preço **por ação**, tratando o caso de cotação por lote de mil (`FATCOT=1000`). **É este o campo a usar no piso de R\$ 1 da máscara.**
- **`FLAG_SITUACAO`** = `True` quando o `CODBDI` indica situação especial no dia.
- Dedup defensivo: 1 registro por (`CODNEG`, data); em duplicata, fica o de maior `VOLTOT`.

In [4]:
df = bruto.copy()

for c in CAMPOS_TEXTO:
    df[c] = df[c].str.strip()

df['DATA_PREGAO'] = pd.to_datetime(df['DATA'].str.strip(), format='%Y%m%d', errors='coerce')

for c in CAMPOS_PRECO_V99:                       # (11)V99 e (16)V99 -> /100
    df[c] = pd.to_numeric(df[c], errors='coerce') / 100.0

for c in ['TOTNEG', 'QUATOT', 'FATCOT']:         # inteiros
    df[c] = pd.to_numeric(df[c], errors='coerce').astype('Int64')

df['PTOEXE'] = pd.to_numeric(df['PTOEXE'], errors='coerce') / 1_000_000.0   # (07)V06
df['DATVEN'] = pd.to_datetime(df['DATVEN'].str.strip(), format='%Y%m%d', errors='coerce')

# --- derivadas ---
df['PRECO_UNIT']    = df['PREULT'] / df['FATCOT'].replace(0, pd.NA)   # preco por acao (trata lote de mil)
df['FLAG_SITUACAO'] = df['CODBDI'].isin(CODBDI_SITUACAO)
df['ANO']           = df['DATA_PREGAO'].dt.year

# periodo
df = df[df['ANO'].between(ANO_INI, ANO_FIM)].copy()

# dedup (CODNEG + data), mantendo maior volume financeiro
antes = len(df)
df = (df.sort_values(['CODNEG', 'DATA_PREGAO', 'VOLTOT'])
        .drop_duplicates(['CODNEG', 'DATA_PREGAO'], keep='last'))
dups = antes - len(df)
if dups:
    print(f"Aviso: {dups:,} registro(s) duplicado(s) (CODNEG+data) resolvidos por maior VOLTOT.")

df = df.sort_values(['CODNEG', 'DATA_PREGAO']).reset_index(drop=True)
print(f"Registros apos tipagem/dedup/periodo: {len(df):,}")
print(f"Periodo: {df['DATA_PREGAO'].min().date()} -> {df['DATA_PREGAO'].max().date()}")
print(f"Tickers distintos: {df['CODNEG'].nunique():,}")

Registros apos tipagem/dedup/periodo: 1,329,429
Periodo: 2010-01-04 -> 2025-12-30
Tickers distintos: 893


## 4. Gravação por ticker (`{TICKER}/{TICKER}.csv` e `.parquet`)

Cada ticker (`CODNEG`) recebe uma pasta própria dentro de `PASTA_SAIDA`. O CSV sai em `utf-8-sig` (abre com acento no Excel).

> Lembrete: tickers que trocaram de código (ex.: `GOLL4`/`GOLL54`) viram **pastas separadas** — por desenho. A reconciliação via `CODISI` é feita no passo seguinte (construção da máscara), não aqui.

In [5]:
PASTA_SAIDA.mkdir(parents=True, exist_ok=True)

col_ordem = ['DATA_PREGAO','CODNEG','CODBDI','TPMERC','NOMRES','ESPECI','PRAZOT','MODREF',
             'PREABE','PREMAX','PREMIN','PREMED','PREULT','PREOFC','PREOFV',
             'TOTNEG','QUATOT','VOLTOT','PREEXE','INDOPC','DATVEN','FATCOT','PTOEXE','CODISI','DISMES',
             'PRECO_UNIT','FLAG_SITUACAO','ANO']
df = df[col_ordem]

def _safe(t: str) -> str:
    return re.sub(r'[^A-Za-z0-9._-]', '_', (t or '').strip())

ok_parquet = GERAR_PARQUET
if GERAR_PARQUET:
    try:
        import pyarrow  # noqa: F401
    except Exception:
        ok_parquet = False
        print("AVISO: pyarrow nao encontrado -- somente CSV sera gerado. Instale com: pip install pyarrow\n")

n = 0
for ticker, g in df.groupby('CODNEG', sort=True):
    nome = _safe(ticker)
    if not nome:
        continue
    pasta = PASTA_SAIDA / nome
    pasta.mkdir(parents=True, exist_ok=True)
    g.to_csv(pasta / f"{nome}.csv", index=False, encoding='utf-8-sig', date_format='%Y-%m-%d')
    if ok_parquet:
        g.to_parquet(pasta / f"{nome}.parquet", index=False)
    n += 1

print(f"{n:,} ticker(s) gravados em: {PASTA_SAIDA}")

893 ticker(s) gravados em: C:\VSCodeWorkspace\1_TCC_Final\data\B3_COTAHIST_Tratatos


## 5. Auditoria

Confira se as `ESPECI` mantidas cobrem o seu universo, e se os papéis problemáticos (Fictor, Gol, Americanas, Oi, Lupatech…) aparecem com o **flag de situação especial** nos períodos certos — é a validação de que a máscara terá o sinal *point-in-time* que precisamos.

In [6]:
print("=== RESUMO ===")
print(f"Tickers: {df['CODNEG'].nunique():,} | Registros: {len(df):,}")
print(f"Periodo: {df['DATA_PREGAO'].min().date()} -> {df['DATA_PREGAO'].max().date()}\n")

print("ESPECI mantidas (confira a cobertura do seu universo):")
print(df['ESPECI'].value_counts().to_string(), "\n")

sit = df[df['FLAG_SITUACAO']]
print("Pregoes em situacao especial (RJ/concordata/RAET/intervencao), por CODBDI:")
print(sit['CODBDI'].value_counts().to_string())
print(f"\nTickers que passaram por situacao especial: {sit['CODNEG'].nunique()}")
print("Exemplos:", ", ".join(sorted(sit['CODNEG'].unique())[:20]))

=== RESUMO ===
Tickers: 893 | Registros: 1,329,429
Periodo: 2010-01-04 -> 2025-12-30

ESPECI mantidas (confira a cobertura do seu universo):
ESPECI
ON      NM    556574
ON            170646
PN            157941
ON      N1     67513
PN      N1     66036
PN      N2     55630
ON      N2     37008
UNT     N2     30926
PNA            24942
ON  ED  NM     14420
PNB     N1     14111
PNA     N1     13613
ON  EJ  NM     11966
PNB            10500
UNT             7402
ON      MA      6634
PN  EJ  N1      4906
ON  EJ          4725
ON  EJ  N1      4564
PN  EJ          4495
PN  ED  N1      3469
ON  ED  N1      3425
ON  ED          3385
ON *            3350
PN *            3287
ON      MB      2828
PN  ED          2500
ON  EDJ NM      2452
PNA     N2      2028
ON  ES  NM      1762
PN  EJ  N2      1682
PNB     N2      1619
UNT ED  N2      1458
ON  EJ  N2      1403
PN  ED  N2      1402
ON  ED  N2      1176
UNT     N1      1128
ON  ES           889
PNC              854
ON  ATZ NM       848
PNA ED      

In [7]:
for t in ['GOLL4','GOLL54','FICT3','ATOM3','AMER3','OIBR3','LUPA3']:
    g = df[df['CODNEG'] == t]
    if g.empty:
        print(f"{t}: AUSENTE"); continue
    pmin = g.groupby('ANO')['PRECO_UNIT'].min()
    sub1 = list(pmin[pmin < 1.0].index)
    print(f"{t}: {len(g)} pregões | FLAG_SITUACAO já True? {g['FLAG_SITUACAO'].any()} | anos com preço < R$1: {sub1}")

# cobertura do seu universo — cole sua lista dos 118:
# meus_118 = [...]
# print('faltando no COTAHIST:', sorted(set(meus_118) - set(df['CODNEG'].unique())))

GOLL4: 3827 pregões | FLAG_SITUACAO já True? False | anos com preço < R$1: [2024, 2025]
GOLL54: 140 pregões | FLAG_SITUACAO já True? False | anos com preço < R$1: [2025]
FICT3: 256 pregões | FLAG_SITUACAO já True? False | anos com preço < R$1: []
ATOM3: 2279 pregões | FLAG_SITUACAO já True? True | anos com preço < R$1: [2015, 2016]
AMER3: 1113 pregões | FLAG_SITUACAO já True? True | anos com preço < R$1: [2023, 2024]
OIBR3: 3402 pregões | FLAG_SITUACAO já True? True | anos com preço < R$1: [2016, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
LUPA3: 3952 pregões | FLAG_SITUACAO já True? True | anos com preço < R$1: [2013, 2014, 2015, 2016, 2020, 2025]


In [8]:
for t in ['GOLL4','GOLL54','ATOM3','FICT3','OIBR3','LUPA3','AMER3']:
    g = df[df['CODNEG'] == t]
    if g.empty:
        print(f"{t}: AUSENTE"); continue
    isins = g['CODISI'].dropna().unique()
    print(f"{t:7s} | ISIN(s): {list(isins)} | {g['DATA_PREGAO'].min().date()} -> {g['DATA_PREGAO'].max().date()}")

GOLL4   | ISIN(s): ['BRGOLLACNPR4'] | 2010-01-04 -> 2025-06-11
GOLL54  | ISIN(s): ['BRGOLLA01PR5'] | 2025-06-12 -> 2025-12-30
ATOM3   | ISIN(s): ['BRATOMACNOR1'] | 2015-10-14 -> 2024-12-18
FICT3   | ISIN(s): ['BRFICTACNOR3'] | 2024-12-19 -> 2025-12-30
OIBR3   | ISIN(s): ['BROIBRACNOR1'] | 2012-04-09 -> 2025-12-30
LUPA3   | ISIN(s): ['BRLUPAACNOR8'] | 2010-01-04 -> 2025-12-30
AMER3   | ISIN(s): ['BRAMERACNOR6'] | 2021-07-19 -> 2025-12-30
